# Petaling WWTP Effluent Monitoring Explorer

Exploratory summaries of the pinned local final-effluent snapshot for PETALING-MAWAR (PTG248), 2023–2025. No derived GHG estimate or legal compliance assessment is made. `compliance` is the source-recorded status.

- [Data dictionary](../../data/public/DATA_DICTIONARY.md)
- [License and attribution](../../data/public/README.md)
- [Scientific limits](../../docs/science/VALIDATION.md)

Run from the repository root or this notebook directory. The notebook reads local files and makes no network requests.


## 1. Load the versioned snapshot

This reads the CSV snapshot committed with the notebook, not the live database.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/public/manifest.json").is_file())
BASE = ROOT / "data/public"
manifest = json.loads((BASE / "manifest.json").read_text())
samples = pd.read_csv(BASE / "effluent_samples.csv", parse_dates=["sample_date"])
assert len(samples) == manifest["tables"]["effluent_samples"]["rows"]
manifest["dataset"], manifest["version"], len(samples)


## 2. Inspect lab results

Below-detection-limit (`*_bdl`) values are stored at the laboratory detection limit. Treat those values as censored observations; they should not be interpreted as exact concentrations.


In [ ]:
samples = samples.sort_values("sample_date").copy()
samples[["sample_date", "bod", "cod", "tss", "nh3n", "no3n", "compliance"]].describe(include="all")


## 3. Source-recorded status by year

This summarizes only source rows labelled `Comply` or `Not Comply`. It is not a fresh regulatory assessment; confirm status definitions and the applicable site permit before using it for compliance reporting.


In [ ]:
known = samples[samples["compliance"].isin(["Comply", "Not Comply"])].copy()
reported_rate = (
    known.assign(reported_comply=known["compliance"].eq("Comply"))
    .groupby("source_year")["reported_comply"]
    .mean()
    .mul(100)
)
ax = reported_rate.plot(kind="bar", color="#2563eb", figsize=(6, 3.5))
ax.set_ylabel("Samples recorded 'Comply' (%)")
ax.set_xlabel("Source year")
ax.set_ylim(0, 100)
ax.set_title("Source-recorded status summary — PTG248")
for i, value in enumerate(reported_rate):
    ax.text(i, value + 1, f"{value:.0f}%", ha="center")
plt.tight_layout()
plt.show()


## 4. Effluent trends

No universal permit-limit lines are drawn: applicable discharge conditions depend on the receiving-water category, system history, and site permit. Below-detection-limit values are omitted from these simple lines rather than plotted as exact values.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, metric, label in zip(axes, ["bod", "cod"], ["BOD", "COD"]):
    below_limit = samples[f"{metric}_bdl"].fillna(False).astype(bool)
    values = samples[metric].where(~below_limit)
    ax.plot(samples["sample_date"], values, marker=".", linewidth=0.8, color="#0ea5e9")
    ax.set_title(f"{label} (mg/L; non-BDL results)")
    ax.set_ylabel("mg/L")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()


---
### Cite this dataset
See [CITATION.cff](https://github.com/Monash-WWTP/wwtp-dashboard/blob/main/CITATION.cff). For GHG accounting, use the separately documented scenario method only with valid influent and plant operating data; the final-effluent snapshot is not an input dataset for that calculation.
